# Datamine RANDOM Process Example

This notebook demonstrates how to configure and run the **`random`** process wrapper in `dmstudio`.

## Process Description

## RANDOM

Generate random numbers. RANDOM is similar to (and supersedes) the legacy **[MONACO](<monaco.md>)** process.

The type of random distribution is primarily dictated by DISTRIB. Each option supports either one or two parameters (P1 and P2).

* 1Produce random numbers using a **uniform** distribution with minimum **P1** and maximum **P2**.

* 2Produce random numbers using an **exponential** distribution with rate **P1**.

* 3Produce random numbers using a **normal** distribution with mean **P1** and standard deviation **P2**.

* 4Produce random numbers using a **Laplacian** distribution with location **P1** and scale parameter **P2**.

* 5Produce random numbers using a **Weibull** distribution with shape parameter **P1** and scale parameter **P2**.

* 6 Produce random numbers using a **Cauchy** distribution with location **P1** and scale parameter **P2**.

* 7Produce random numbers using a **lognormal** distribution with log mean **P1** and log standard deviation **P2**.

**Note** : you can also generate random numbers using the **[EXTRA](<extra.md>)** process.

### Input Files:

### Output Files:

* **out** (Table):
  Output file containing random values.
  Required=Yes

### Fields:

* **outfield** (Any : IN):
  Field to write the random variables into.
  Default=Undefined
  Required=No

### Parameters:

* **nrecs**:
  Number of records required in output file
  Range=Undefined
  Values=Undefined
  Default=1
  Required=Yes

* **distrib**:
  Type of distribution to use to generate random numbers. See "Distribution Options"
  above.
  Range=1,7
  Values=1,2,3,4,5,6,7
  Default=1
  Required=Yes

* **p1**:
  First parameter of the chosen distribution.
  Range=Undefined
  Values=Undefined
  Default=0
  Required=Yes

* **p2**:
  Second parameter of the chosen distribution. Ignored if **DISTRIB** =2.
  Range=Undefined
  Values=Undefined
  Default=1
  Required=No

* **seed**:
  Random number seed. If the same non-zero seed is used for multiple runs then the same
  set of random numbers will be generated. If **SEED** is zero, subsequent results with
  the same parameters will differ.
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob

import pandas as pd

from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('random')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.copy_database_files(files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute random
print("Running random...")
dm_fil.random(
    out_o='t_random_out',  # required
    outfield_f='optional',  # required
    nrecs_p='required_val',  # required
    distrib_p='required_val',  # required
    ps_f=['optional'],  # required
    # seed_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("random execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_random_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")